# Lesson 02 - Microsoft Agent Framework अन्वेषण गर्दै

**Microsoft Agent Framework (MAF)** एक AI एजेन्टहरू निर्माण गर्नको लागि एक एकीकृत फ्रेमवर्क हो। यसले चार मुख्य निर्माण ब्लकहरूसँग एक सफा, संयोज्य वास्तुकला प्रदान गर्दछ:

- **Client** – AI मोडेल अन्त्यबिन्दुमा जडान गर्छ र सञ्चार ह्यान्डल गर्छ
- **Agent** – निर्देशनहरू र उपकरण परिभाषाहरूका साथ क्लाइन्टलाई आवरण गर्छ
- **Tools** – मोडेलले कल गर्न सक्ने कस्टम कार्यहरूसँग एजेन्ट क्षमताहरू विस्तार गर्छ
- **Session** – बहु-पटक अन्तर्क्रियाहरूको लागि संवाद इतिहासलाई कायम राख्छ

यस पाठमा, हामी यी अवधारणाहरू प्रयोग गरेर गन्तव्य उपलब्धता जाँच गर्ने **यात्रा बुकिङ एजेन्ट** निर्माण गर्नेछौं।


## सेटअप


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## एजेन्ट फ्रेमवर्क आर्किटेक्चर बुझ्ने

माइक्रोसफ्ट एजेन्ट फ्रेमवर्कले तहगत आर्किटेक्चर अनुसरण गर्दछ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **क्लाइन्ट** – एक `AzureAIProjectAgentProvider` Azure OpenAI डिप्लोयमेन्टसँग जडान हुन्छ। यसले प्रमाणिकरण, अनुरोध ढाँचा बनाउने, र प्रतिक्रिया पार्सिङ्गलाई सम्हाल्छ।
2. **एजेन्ट** – क्लाइन्टबाट `provider.create_agent()` मार्फत सिर्जना गरिएको, एजेन्टले मोडल पहुँचलाई निर्देशनहरूसँग (सिस्टम प्रॉम्प्ट) र उपकरणहरूसँग संयोजन गर्छ।
3. **उपकरणहरू** – Python का कार्यहरू जुन `@tool` ले सजाइएको हुन्छन् र जसलाई एजेन्टले क्रियाहरू गर्ने वा डेटा प्राप्त गर्न बोलाउन सक्छ।
4. **सेसन** – एक `AgentSession` वस्तु (जे `agent.create_session()` बाट सिर्जना गरिन्छ) जसले संवाद इतिहास भण्डारण गर्छ, बहु-पटकको संवाद सक्षम पार्दै जहाँ एजेन्टले अघिल्लो प्रस context्ग सम्झन्छ।

हामी प्रत्येक तहलाई चरणबद्ध रूपमा निर्माण गरौं।


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool डेकोरेटरसँग उपकरणहरू थप्दै

उपकरणहरूले एजेन्टहरूले टेक्स्ट उत्पादनभन्दा बाहिर क्रियाहरू लिन अनुमति दिन्छ। `@tool` डेकोरेटरले सामान्य Python फङ्सनलाई एजेन्टले कल गर्न सक्ने केहि कुरा मा रूपान्तरण गर्छ।

मुख्य बुँदाहरू:
- मोडललाई हरेक प्यारामिटर बुझाउन `Annotated[type, "description"]` प्रयोग गर्नुहोस्।
- डकस्ट्रिङले मोडलले देख्ने उपकरणको विवरण बनेको हुन्छ।
- `approval_mode="never_require"` को मतलब उपकरण स्वचालित रूपमा प्रयोगकर्ताको पुष्टि बिना चल्छ।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## उपकरणहरूसँग एजेन्ट बनाउँदै

अब हामी क्लाइन्ट, निर्देशनहरू, र उपकरणहरूलाई एउटा एजेन्टमा एकीकृत गर्दछौं। `निर्देशनहरू` सिस्टम प्रॉम्प्टको रूपमा काम गर्छन् — तिनीहरूले एजेन्टको व्यक्तित्व र व्यवहार परिभाषित गर्छन्।


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## सत्रहरू सहित बहु-चरित्र संवादहरू

एक `AgentSession` (जिसलाई `agent.create_session()` मार्फत सिर्जना गरिन्छ) संवादको सबै सन्देशहरूलाई ट्र्याक गर्छ। हरेक `agent.run()` कलमा उही सत्र पास गरेर, एजेन्टले पूर्ण संवाद इतिहासमा पहुँच राख्छ र पहिलेका सन्देशहरूमा पुनः सन्दर्भ गर्न सक्छ।

हामी `tools=[check_destination_availability]` पास गर्छौं ताकि एजेन्टले हरेक चरित्रमा हाम्रो उपलब्धता जाँचकर्ता कल गर्न सकून्।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## सारांश

यस पाठमा तपाईंले Microsoft Agent Framework का चार स्तम्भहरू अन्वेषण गर्नुभयो:

| अवधारणा | तपाईंले सिक्नुभयो के |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` प्रमाणपत्र-आधारित प्रमाणीकरणसँग Azure OpenAI मा जडान गर्दछ |
| **Agent** | `provider.create_agent()` एक मोडेल जडानलाई निर्देशन र नामसँग बन्द गर्दछ |
| **Tools** | `@tool` डेकोरेटरले एजेन्टलाई कल गर्न पाइने Python कार्यहरू खोल्दछ |
| **Session** | `agent.create_session()` बहु चरणहरूको वार्तालाप इतिहास कायम राख्दछ |

यी संरचनात्मक अंशहरू मिलेर एजेन्टहरू सिर्जना गर्छन् जसले प्राकृतिक कुराकानी गर्न सक्छन्, बाह्य कार्यहरू कल गर्न सक्छन्, र सन्दर्भ कायम राख्न सक्छन् — यो पछि पाठहरूमा थप उन्नत एजेन्टिक ढाँचाहरूको आधार हो।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धताको प्रयास गर्छौं भने पनि, कृपया जानकार हुनुहोस् कि स्वचालित अनुवादमा त्रुटि वा गलतफहमी हुन सक्छ। मूल दस्तावेज यसको मूल भाषामा नै आधिकारिक स्रोत मानिनेछ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा व्याख्याको लागि हामी दायित्व बहन गर्दैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
